# 02 - Real-data pilot: an E7.5 mouse embryo panel

Same code paths as notebook 01, real data.

**Query.** Xie et al. 2025 Stereo-seq whole-embryo atlas, E7.5 replicate 1
([GSE278603](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278603)), reduced to the
4-gene panel plus the 2D sagittal coordinate - the honest simulation of "we could only measure
k genes".

**Candidate set D** (`panelprep/io/catalog.py`), each with the brief's category tag:

| category | entries |
|---|---|
| `spatial` | the other five embryos of the same study (E7.5 / E7.75 / E8.0) |
| `single_cell` | Mittnenzweig et al. 2021 MARS-seq ([GSE169210](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE169210)), at E7.5 **and** at E6.5 |
| `causal` | the Foxc1/Foxc2 double-knockout chimera arm of the same study |
| `non_transcriptomic` | ESM-2 sequence and STRING PPI sources (no cell x gene matrix) |

The two Mittnenzweig entries are a **built-in control**: with an E7.5 query, the stage-matched
reference should agree better than the E6.5 one. If it does not, the scores are not measuring
developmental similarity.

**Normalisation, stated up front.** The Stereo-seq `X` is `log1p` of *SCT-corrected* counts whose
per-cell totals are not constant, so the loader inverts the log, renormalises each cell to CP10K
and re-applies `log1p`. MARS-seq UMIs go straight to `log1p(CP10K)`. Both end up on the same scale;
the SCT variance stabilisation upstream is recorded in `meta['normalization_note']` and is not
undone.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # import panelprep from the repo root
os.chdir(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, pandas as pd
from panelprep import Config
print('panelprep ready; cwd =', os.getcwd())

In [ ]:
from panelprep.io.catalog import catalog_table
catalog_table()

## The input the preprocessing team is handed

In [ ]:
from panelprep.io.loaders import query_from_seu3d

PANEL = ['T', 'Krt8', 'Mest', 'Dnmt3b']   # MODIFY HERE: the measured gene subset
DESCRIPTOR = (
    'Spatial transcriptomics of a whole mouse embryo at embryonic day 7.5 (E7.5) during '
    'gastrulation and early organogenesis. Cells retain tissue coordinates. We study germ layer '
    'patterning and the emergence of mesoderm and endoderm progenitors.')

cfg = Config.from_yaml('configs/real_pilot.yaml')
query = query_from_seu3d('GSM9046243_Embryo_E7.5_stereo_rep1.h5ad', PANEL, DESCRIPTOR,
                         max_obs=4000, rng=np.random.default_rng(cfg.seed),
                         name='query_E7.5_rep1')
print(query.expression.shape, query.coords.shape, query.units)
print('detection rate per panel gene:', (query.expression > 0).mean().round(3).to_dict())
query.expression.describe().T.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(PANEL), figsize=(4 * len(PANEL), 3.6), constrained_layout=True)
for ax, g in zip(np.atleast_1d(axes), PANEL):
    s = ax.scatter(query.coords[:, 0], query.coords[:, 1], c=query.expression[g], s=3,
                   cmap='magma', vmin=0)
    ax.set_title(f'{g}  (det. {float((query.expression[g] > 0).mean()):.1%})')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(s, ax=ax, shrink=0.8, label='log1p CP10K')
fig.suptitle('Query: E7.5 embryo, sagittal projection, measured panel only')
fig.savefig('outputs/real_pilot/figures/query_panel.png', dpi=150) if os.path.isdir('outputs/real_pilot/figures') else None
fig

## Stage 1a - descriptor match

In [ ]:
from panelprep.pipeline import match_candidates
match_candidates(query, cfg)

## Load the candidate set

The `non_transcriptomic` and `kegg_regulatory` entries load with no expression matrix on purpose - they are embedding sources, and the scores below correctly mark them not applicable.

In [ ]:
from panelprep.pipeline import load_candidates
candidates = load_candidates(cfg)
pd.DataFrame([{'name': n, 'category': c.category, 'obs_unit': c.obs_unit,
               'n_obs': c.n_obs, 'n_genes': len(c.genes), 'coords': c.has_coords,
               'units': c.units} for n, c in candidates.items()]).set_index('name')

## Stage 1b - agreement scores and the comparison table

In [ ]:
from panelprep.pipeline import score_candidates
results = score_candidates(query, candidates, cfg)
results.pivot_table(index='dataset', columns='score', values='value', dropna=False).round(4)

In [ ]:
from panelprep.viz import score_heatmap
score_heatmap(results, out_path='outputs/real_pilot/figures/score_heatmap.png',
              title=f'E7.5 panel ({", ".join(PANEL)}) vs candidate set D')

### The stage control

Read the two Mittnenzweig rows against each other. Same study, same platform, same processing -
the only difference is developmental age, so any score that is measuring developmental similarity
rather than platform must rank E7.5 above E6.5.

In [ ]:
piv = results.pivot_table(index='dataset', columns='score', values='value', dropna=False)
ctrl = piv.loc[[i for i in ['mittnenzweig_wt_E7.5', 'mittnenzweig_wt_E6.5'] if i in piv.index]]
display(ctrl.round(4))
from panelprep.scores import SCORES
for s in ctrl.columns:
    if ctrl[s].isna().any():
        continue
    better = ctrl[s].idxmin() if SCORES[s].meta['orientation'] == 'lower_is_better' else ctrl[s].idxmax()
    print(f'{s:30s} favours {better}')

## Stages 2-4: gene embeddings, annotations, report

The gene universe is the genes shared across the expression candidates - not the measured panel. That is the point of item 0: a representation for genes that were never measured.

In [ ]:
from panelprep.pipeline import run_pipeline
out = run_pipeline(query, cfg, candidates=candidates)
print('gene universe:', len(out.get('gene_universe') or []))
print('report:', out.get('report_path'))
pd.DataFrame([{'embedding': k, 'category': v.category, 'genes': v.matrix.shape[0],
               'dim': v.dim, 'nan_genes': int(v.matrix.isna().all(axis=1).sum())}
              for k, v in (out.get('embeddings') or {}).items()]).set_index('embedding')

In [ ]:
ann = out.get('annotation')
if ann is not None:
    display(ann.table.head(10))
    print({c: int(ann.table[c].sum()) for c in ['is_ligand', 'is_receptor', 'is_tf']})
    print('genes with a pathway:', int((ann.table['top_pathway'].astype(str) != '').sum()))

In [ ]:
from panelprep.viz import embedding_scatter, embedding_quality
embs = out.get('embeddings') or {}
rows = []
for name, emb in embs.items():
    if ann is not None:
        rows.append({'embedding': name, **embedding_quality(emb, ann)})
pd.DataFrame(rows).set_index('embedding') if rows else 'no annotation available'